# IndyCar PDF scraper

In [ ]:
!pip install pypdf pdfplumber requests pandas

In [ ]:
import re
import pandas as pd
import pdfplumber
import requests
import os
import time

from pathlib import Path
from datetime import timedelta

In [ ]:
def split_name(full_name: str) -> tuple:
    full_name = full_name.strip().replace(" (R)", "").strip()
    if "," in full_name:
        parts = full_name.split(",", 1)
        last  = parts[0].strip()
        first = parts[1].strip()
    else:
        parts = full_name.split()
        first = parts[0] if parts else ""
        last  = " ".join(parts[1:]) if len(parts) > 1 else ""
    return first, last

def extract_text(pdf_path: str) -> str:
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += (page.extract_text() or "") + "\n"
    return text

In [ ]:
# Race Result Parser - Most Important Data
def parse_race_result_file(text:str) -> dict:
  result = {}

  m = re.search(r"Caution Laps:\s+(\d+)", text)
  result["TotalCautionLaps"] = int(m.group(1)) if m else 0

  m = re.search(r"Lead Changes:\s+(\d+)", text)
  result["TotalLeadChanges"] = int(m.group(1)) if m else 0

  m = re.search(r"End of Lap\s+(\d+):", text)
  result["TotalRaceLaps"] = int(m.group(1)) if m else 0

  m = re.search(r"([\d.]+)\s+mile\(s\)", text)
  result["TrackLength"] = float(m.group(1)) if m else 0.0

  m = re.search(r"Avg Speed:\s+(\d+)", text)
  result["AvgRaceSpeed"] = int(m.group(1)) if m else 0

  m = re.search(r"Fastest Lap:\s+([\d.]+)", text)
  result["FastestLapSpeed"] = float(m.group(1)) if m else 0.0

  section = text[text.find("Caution Summary"):] if "Caution Summary" in text else ""
  caution = re.compile(
      r"(\d+)\s+to\s+(\d+)\s+(\d+)\s+([A-Z][^\n]+)"
  )

  cautions = []
  for m in caution.finditer(section):
    cautions.append({
        "start_lap": int(m.group(1)),
        "end_lap": int(m.group(2)),
        "duration": int(m.group(3)),
        "reason": m.group(4).split(":")[0].strip(),
    })

  result["TotalCautions"] = len(cautions)
  result["Cautions"] = cautions

  return result

In [ ]:
# Pit Summary Parser - Important Data
def parse_pit_summary_file(text: str) -> list[dict]:
  pit_stops = []

  drivers = re.compile(
      r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+(\d+)\s+Pit Stops"
  )

  stops = re.compile(
      r"^(\d+)\s+(\d+)\s+(\d+)\s+(\d{2}:\d{2}:\d{2}\.\d+)$"
  )

  driver = None

  for line in text.split("\n"):
    dm = drivers.search(line)
    if dm:
      first, last = split_name(dm.group(3))
      driver = {
          "carNum": int(dm.group(2)),
          "FName":first,
          "LName": last,
          "TotalPitStops": int(dm.group(5)),
      }
      continue

    sm = stops.match(line.strip())
    if sm and driver:
      pit_stops.append({
          **driver,
          "StopNumber": int(sm.group(1)),
          "DriverLap": int(sm.group(2)),
          "RaceLap": int(sm.group(3)),
          "Time": sm.group(4),
      })

  return pit_stops

In [ ]:
# Lap Chart Parser
def parse_lap_chart_file(text: str) -> dict:
  drivers = re.compile(
      r"(\d{1,2})\s+-\s+(.+?)\s+\((\d+)\)\s+(\d+)\s+([\d\s]+)"
  )

  driver_info = {}
  positions = []

  for line in text.split("\n"):
    m = drivers.match(line)
    if m:
      car_num = int(m.group(1))
      first, last = split_name(m.group(2))
      position =  int(m.group(4))
      car_nums = [int(x) for x in m.group(5).strip().split()]

      if car_num not in driver_info:
        driver_info[car_num] = (first, last)

      positions.append((position, car_nums))

  # Find what position each car is at
  car_positions = {car_num: [] for car_num in driver_info}

  #rows by position
  from collections import defaultdict
  position_groups = defaultdict(list)
  for position, car_nums in positions:
    position_groups[position].append(car_nums)

  # Check all pages
  all_pages = {}
  for position, groups in position_groups.items():
    all_pages[position] = []
    for group in groups:
      all_pages[position].extend(group)


  total_laps = max(len(v) for v in all_pages.values())

  # Find positions per lap
  for lap_i in range(total_laps):
    for position, car_nums in all_pages.items():
      if lap_i < len(car_nums):
        car_num = car_nums[lap_i]
        if car_num in car_positions and len(car_positions[car_num]) == lap_i:
          car_positions[car_num].append(position)

  return {
      car_num: {
          "DriverName": driver_info[car_num],
          "Positions": car_positions[car_num],
      }
      for car_num in driver_info
  }

In [ ]:
# Section Results Parser
def parse_section_data(text: str) -> list[dict]:
    records = []
    current_car = None
    current_driver = None
    next_lap = None
    seen = set()

    for line in text.split("\n"):
        dm = re.match(r"Section Data for Car (\d+) - (.+)", line)
        if dm:
            current_car = int(dm.group(1))
            current_driver = dm.group(2).strip()
            continue

        lm = re.match(r"^(\d+)$", line.strip())
        if lm and current_car is not None:
            next_lap = int(lm.group(1))
            continue

        parts = line.strip().split()
        if parts and parts[0] == "T" and current_car is not None and next_lap is not None:
            try:
                vals = [float(x) for x in parts[1:]]
            except ValueError:
                next_lap = None
                continue
            n = len(vals)

            if n < 3:
                next_lap = None
                continue

            key = (current_car, next_lap)
            if key in seen:
                next_lap = None
                continue
            seen.add(key)

            # Detect: last value is much smaller than second-to-last
            if n >= 4 and vals[-1] < vals[-2] and vals[-1] < vals[-2] * 0.6:
                lap_time = vals[-2]
            else:
                lap_time = vals[-1]

            first, last = split_name(current_driver)
            records.append({
                "car": current_car,
                "FName": first,
                "LName": last,
                "lap": next_lap,
                "lap_time": lap_time,
            })
            next_lap = None

    return records

In [ ]:
def parse_qualy(text: str) -> list[dict]:
    results = []
    pattern = re.compile(
        r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+"
        r"(\d{2}:\d{2}\.\d+)\s+([\d.]+)\s+(.+)"
    )

    for line in text.split("\n"):
      m = pattern.match(line.strip())
      if m:
        first, last = split_name(m.group(3))
        results.append({
            "carNum": int(m.group(2)),
            "FName": first,
            "LName": last,
            "Rank": int(m.group(1)),
            "QualyTime": m.group(5),
            "QualySpeed": float(m.group(6)),
            "Session": m.group(7).strip(),
        })

    return results

def parse_practice(text: str) -> list[dict]:
    results = []
    pattern = re.compile(
        r"(\d{1,2})\s+(\d{1,2})\s+(.+?)\s+(D/[CH]/F)\s+"
        r"(?:Practice \d.*?\s+)?"
        r"(\d{2}:\d{2}\.\d+)\s+([\d.]+)\s+.*?(\d+)\s*$"
    )
    for line in text.split("\n"):
        m = pattern.match(line.strip())
        if m:
            first, last = split_name(m.group(3))
            results.append({
                "carNum": int(m.group(2)),
                "FName": first,
                "LName": last,
                "Rank": int(m.group(1)),
                "bestTime": m.group(5),
                "Speed": float(m.group(6)),
                "totalLaps": int(m.group(7)),
            })
    return results

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
## Dowloads report files of each race

df = pd.read_csv("IndyCar_dataset_v23.csv")
df["EventDate"] = pd.to_datetime(df["EventDate"])

# Get one row per event

events = (df[["EventID", "Track", "EventDate"]]
          .drop_duplicates("EventID")
          .assign(EventDate=lambda x: pd.to_datetime(x["EventDate"]))
          .sort_values("EventDate")
          .reset_index(drop=True))

# Filter to only years with available PDFs
events = events[(events["EventDate"].dt.year >= 2013) &
                (events["EventDate"].dt.year <= 2024)]

In [ ]:
BASE_URL = "http://www.imscdn.com/INDYCAR/Documents"
BASE_DIR = Path("/content/drive/MyDrive/indycar/races")
BASE_DIR.mkdir(exist_ok=True)

def get_filenames(year):
    results_file = "indycar-race-lapfinal.pdf" if year <= 2014 else "indycar-race-results.pdf"
    return {
        "results": [results_file],
        "event_summary": ["indycar-eventsummary.pdf"],
        "pit_stops": ["indycar-race-pitstopsummary.pdf"],
        "lap_chart": ["indycar-race-lapchart.pdf"],
        "section_data": ["indycar-sectionresults-race.pdf"],
    }

def download_pdf(url, dest_path):
    """Download a single PDF with error handling."""
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            with open(dest_path, "wb") as f:
                f.write(response.content)
            return True
        else:
            print(f"  ❌ {response.status_code} — {url}")
            return False
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return False

def get_date_candidates(race_date):
    return [
        race_date,
        race_date - timedelta(days=1),
        race_date - timedelta(days=2),
    ]
# --- Main download loop ---
total_downloaded = 0
total_failed     = 0

for i, row in events.iterrows():
    event_id   = row["EventID"]
    event_date = row["EventDate"]
    track      = row["Track"]
    year       = event_date.year

    # Build folder name matching our structure
    track_clean = (track
                   .replace(" ", "_")
                   .replace(".", "")
                   .replace("/", "_")
                   .replace("(", "")
                   .replace(")", "")
                   .lower())
    folder_name = f"{str(i+1).zfill(3)}_{track_clean}_{year}"
    folder_path = BASE_DIR / folder_name
    folder_path.mkdir(exist_ok=True)

    date_str   = event_date.strftime("%Y-%m-%d")
    filenames  = get_filenames(year)

    print(f"\n📥 {folder_name}")

    for doc_type, candidates in filenames.items():
      downloaded = False
      date_candidates = get_date_candidates(event_date)

      for filename in candidates:
        if downloaded:
          break
        dest_path = folder_path / filename

        # Skip if already downloaded
        if dest_path.exists():
            print(f"  ✓ Already exists — {filename}")
            downloaded = True
            break

        for try_date in date_candidates:
          date_str = try_date.strftime("%Y-%m-%d")
          url = f"{BASE_URL}/{event_id}/{date_str}/{filename}"
          success = download_pdf(url, dest_path)

          if success:
              print(f"  ✅ {filename}")
              total_downloaded += 1
              downloaded = True
              break

          time.sleep(1)

    if not downloaded:
        print(f"  ❌ No file found for {doc_type}")
        total_failed += 1

    time.sleep(3)  # be polite to the server

print(f"\n✅ Done! Downloaded: {total_downloaded} | Failed: {total_failed}")

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/indycar/races")
OUTPUT_DIR = Path("/content/drive/MyDrive/indycar/scraped_data")
OUTPUT_DIR.mkdir(exist_ok=True)

# Load existing progress if restarting
if (OUTPUT_DIR / "race_metadata.csv").exists():
    all_race_metadata = pd.read_csv(OUTPUT_DIR / "race_metadata.csv").to_dict("records")
    all_cautions = pd.read_csv(OUTPUT_DIR / "cautions.csv").to_dict("records")
    all_pit_stops = pd.read_csv(OUTPUT_DIR / "pit_stops.csv").to_dict("records")
    all_lap_positions = pd.read_csv(OUTPUT_DIR / "lap_positions.csv").to_dict("records")
    all_lap_times = pd.read_csv(OUTPUT_DIR / "lap_times.csv").to_dict("records")
    processed = {r["event_id"] for r in all_race_metadata}
    print(f"Resuming — {len(processed)} races already done")
else:
    all_race_metadata = []
    all_cautions = []
    all_pit_stops = []
    all_lap_positions = []
    all_lap_times = []
    processed = set()

folders = sorted([f for f in BASE_DIR.iterdir() if f.is_dir()])

for i, folder in enumerate(folders):
    event_id = folder.name

    if event_id in processed:
        continue

    print(f"[{i+1}/{len(folders)}] Processing {event_id}...")

    # --- Race Results ---
    race_results_path = folder / "indycar-race-results.pdf"
    if not race_results_path.exists():
        race_results_path = folder / "indycar-race-lapfinal.pdf"
    if not race_results_path.exists():
        matches = list(folder.glob("indycar-race-results*.pdf")) or list(folder.glob("indycar-race-lapfinal*.pdf"))
        race_results_path = matches[0] if matches else race_results_path
    if race_results_path.exists():
        try:
            text = extract_text(str(race_results_path))
            data = parse_race_result_file(text)
            cautions = data.pop("Cautions", [])
            data["event_id"] = event_id
            all_race_metadata.append(data)
            for c in cautions:
                c["event_id"] = event_id
                all_cautions.append(c)
        except Exception as e:
            print(f"  ❌ Race results failed: {e}")

    # --- Pit Stop Summary ---
    pit_path = folder / "indycar-race-pitstopsummary.pdf"
    if not pit_path.exists():
        matches = list(folder.glob("indycar-race-pitstopsummary*.pdf"))
        pit_path = matches[0] if matches else pit_path
    if pit_path.exists():
        try:
            text = extract_text(str(pit_path))
            stops = parse_pit_summary_file(text)
            for s in stops:
                s["event_id"] = event_id
                all_pit_stops.append(s)
        except Exception as e:
            print(f"  ❌ Pit stops failed: {e}")

    # --- Lap Chart ---
    lap_chart_path = folder / "indycar-race-lapchart.pdf"
    if not lap_chart_path.exists():
        matches = list(folder.glob("indycar-race-lapchart*.pdf"))
        lap_chart_path = matches[0] if matches else lap_chart_path
    if lap_chart_path.exists():
        try:
            text = extract_text(str(lap_chart_path))
            data = parse_lap_chart_file(text)
            for car, info in data.items():
                first, last = info["DriverName"]
                for lap_idx, position in enumerate(info["Positions"], start=1):
                    all_lap_positions.append({
                        "event_id": event_id,
                        "car": car,
                        "FName": first,
                        "LName": last,
                        "lap": lap_idx,
                        "position": position,
                    })
        except Exception as e:
            print(f"  ❌ Lap chart failed: {e}")

    # --- Section Data (Lap Times) ---
    section_path = folder / "indycar-sectionresults-race.pdf"
    if not section_path.exists():
        matches = list(folder.glob("indycar-sectionresults-race*.pdf"))
        section_path = matches[0] if matches else section_path
    if section_path.exists():
        try:
            print(f"  Parsing section data...")
            text = extract_text(str(section_path))
            records = parse_section_data(text)
            for r in records:
                r["event_id"] = event_id
                all_lap_times.append(r)
        except Exception as e:
            print(f"  ❌ Section data failed: {e}")

    # Save all 5 CSVs after every race
    pd.DataFrame(all_race_metadata).to_csv(OUTPUT_DIR / "race_metadata.csv", index=False)
    pd.DataFrame(all_cautions).to_csv(OUTPUT_DIR / "cautions.csv", index=False)
    pd.DataFrame(all_pit_stops).to_csv(OUTPUT_DIR / "pit_stops.csv", index=False)
    pd.DataFrame(all_lap_positions).to_csv(OUTPUT_DIR / "lap_positions.csv", index=False)
    pd.DataFrame(all_lap_times).to_csv(OUTPUT_DIR / "lap_times.csv", index=False)

print(f"\n✅ Done!")
print(f"  race_metadata: {len(all_race_metadata)} races")
print(f"  cautions: {len(all_cautions)} caution periods")
print(f"  pit_stops: {len(all_pit_stops)} pit stop records")
print(f"  lap_positions: {len(all_lap_positions)} position records")
print(f"  lap_times: {len(all_lap_times)} lap time records")

In [ ]:
df_positions = pd.read_csv(OUTPUT_DIR / "lap_positions.csv").drop_duplicates()
df_positions.to_csv(OUTPUT_DIR / "lap_positions.csv", index=False)

df_times = pd.read_csv(OUTPUT_DIR / "lap_times.csv").drop_duplicates()
df_times.to_csv(OUTPUT_DIR / "lap_times.csv", index=False)

In [ ]:
for f in Path("/content/drive/MyDrive/indycar/scraped_data").glob("*.csv"):
    df = pd.read_csv(f)
    print(f"{f.name}: {len(df)} rows, {len(df.columns)} cols")